In [3]:
import torch
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import v2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import glob
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score

# YOLO

In [8]:
!unzip -q "/content/drive/MyDrive/Colab_Notebooks/archive.zip" -d "/content/"

In [9]:
data_yaml = """
path: /content/dataset
train: images/train
val: images/val

names:
  0: "Road sign"
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

In [10]:
model = YOLO("yolo26n-seg.pt")
results = model.train(data="/content/data.yaml", epochs=40, batch=64 ,imgsz=640, workers=2)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

# IoU, Precision, Recall, L2

In [17]:
def find_images(images_dir):
    files = []
    for f in os.listdir(images_dir):
        if f.lower().endswith((".jpg")):
            files.append(os.path.join(images_dir, f))
    return sorted(files)

def image_to_label_path(img_path, labels_dir):
    base = os.path.splitext(os.path.basename(img_path))[0]
    return os.path.join(labels_dir, base + ".txt")


def load_image_shape(img_path):
    img = cv2.imread(img_path)
    return img.shape


def polygons_txt_to_mask(label_path, img_h, img_w):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)

    if not os.path.exists(label_path):
        return mask

    with open(label_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()

        nums = list(map(float, parts))
        coords = nums[1:]

        polygon = np.array(coords, dtype=np.float32).reshape(-1, 2)
        polygon[:, 0] *= img_w
        polygon[:, 1] *= img_h
        polygon = np.round(polygon).astype(np.int32)

        polygon[:, 0] = np.clip(polygon[:, 0], 0, img_w - 1)
        polygon[:, 1] = np.clip(polygon[:, 1], 0, img_h - 1)

        cv2.fillPoly(mask, [polygon], 1)

    return mask


def prediction_to_mask(result, img_h, img_w):
    if result.masks is None or result.masks.data is None or len(result.masks.data) == 0:
        return np.zeros((img_h, img_w), dtype=np.uint8)

    pred_masks = result.masks.data
    pred_union = torch.any(pred_masks > 0.5, dim=0).cpu().numpy().astype(np.uint8)

    if pred_union.shape != (img_h, img_w):
        pred_union = cv2.resize(
            pred_union,
            (img_w, img_h),
            interpolation=cv2.INTER_NEAREST
        )

    pred_union = (pred_union > 0).astype(np.uint8)
    return pred_union


def compute_metrics(pred_mask, true_mask):
    pred_bool = pred_mask.astype(bool)
    true_bool = true_mask.astype(bool)

    intersection = np.logical_and(pred_bool, true_bool).sum()
    union = np.logical_or(pred_bool, true_bool).sum()

    if union == 0:
        iou = 1.0
    else:
        iou = intersection / union

    pred_flat = pred_mask.flatten()
    true_flat = true_mask.flatten()

    precision = precision_score(true_flat, pred_flat, zero_division=1)
    recall = recall_score(true_flat, pred_flat, zero_division=1)

    l2 = np.sqrt(np.sum((pred_mask.astype(np.float32) - true_mask.astype(np.float32)) ** 2))

    return iou, precision, recall, l2

In [18]:
VAL_IMAGES_DIR = "/content/dataset/images/val"
VAL_LABELS_DIR = "/content/dataset/labels/val"
model = YOLO("/content/runs/segment/train/weights/best.pt")

image_paths = find_images(VAL_IMAGES_DIR)
rows = []

for img_path in tqdm(image_paths, desc="Evaluating val"):
    label_path = image_to_label_path(img_path, VAL_LABELS_DIR)

    shape = load_image_shape(img_path)
    img_h, img_w = shape[:2]

    true_mask = polygons_txt_to_mask(label_path, img_h, img_w)

    results = model.predict(
        source=img_path,
        verbose=False
    )
    result = results[0]
    pred_mask = prediction_to_mask(result, img_h, img_w)

    iou, precision, recall, l2 = compute_metrics(pred_mask, true_mask)

    rows.append({
        "image": os.path.basename(img_path),
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "l2": l2
    })

df = pd.DataFrame(rows)

mean_iou = df["iou"].mean()
mean_precision = df["precision"].mean()
mean_recall = df["recall"].mean()
mean_l2 = df["l2"].mean()

pct_iou_05 = (df["iou"] >= 0.5).mean() * 100
pct_iou_075 = (df["iou"] >= 0.75).mean() * 100
pct_iou_09 = (df["iou"] >= 0.9).mean() * 100

print("\n===== VAL METRICS =====")
print(f"Images count           : {len(df)}")
print(f"Mean IoU               : {mean_iou:.4f}")
print(f"Mean Precision         : {mean_precision:.4f}")
print(f"Mean Recall            : {mean_recall:.4f}")
print(f"Mean L2                : {mean_l2:.4f}")
print(f"Percent IoU >= 0.50    : {pct_iou_05:.2f}%")
print(f"Percent IoU >= 0.75    : {pct_iou_075:.2f}%")
print(f"Percent IoU >= 0.90    : {pct_iou_09:.2f}%")

Evaluating val: 100%|██████████| 127/127 [01:00<00:00,  2.09it/s]


===== VAL METRICS =====
Images count           : 127
Mean IoU               : 0.5623
Mean Precision         : 0.6864
Mean Recall            : 0.7664
Mean L2                : 81.4407
Percent IoU >= 0.50    : 69.29%
Percent IoU >= 0.75    : 7.87%
Percent IoU >= 0.90    : 0.00%


# Проверка на своих изображениях

In [6]:
img_dir = "/content/my_img/"
images = [f"{img_dir}{i}.jpg" for i in range(1, 17)]
results = model(images)


0: 640x480 (no detections), 4.7ms
1: 640x480 5 Road signs, 4.7ms
2: 640x480 1 Road sign, 4.7ms
3: 640x480 6 Road signs, 4.7ms
4: 640x480 4 Road signs, 4.7ms
5: 640x480 10 Road signs, 4.7ms
6: 640x480 1 Road sign, 4.7ms
7: 640x480 12 Road signs, 4.7ms
8: 640x480 6 Road signs, 4.7ms
9: 640x480 4 Road signs, 4.7ms
10: 640x480 4 Road signs, 4.7ms
11: 640x480 3 Road signs, 4.7ms
12: 640x480 3 Road signs, 4.7ms
13: 640x480 6 Road signs, 4.7ms
14: 640x480 1 Road sign, 4.7ms
15: 640x480 2 Road signs, 4.7ms
Speed: 1.9ms preprocess, 4.7ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)


In [11]:
for result in results:
    plt.figure(figsize=(15, 15))
    plt.imshow(result.plot(line_width=2)[..., ::-1])
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.